In [3]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import pandas as pd
import lightgbm as lgb
import json

# Paths setup
silver_path = Path("./ieee_fraud/silver")
unified_train_file = silver_path / "ieee_train_split_merged.parquet"

print("Loading the FULL unified training dataset (590,540 rows)...")
# Read the complete dataset directly from the Silver layer
train_df = pd.read_parquet(unified_train_file)

# Define target and metadata columns to drop from features
target_col = "isFraud"
drop_cols = {target_col, "TransactionID", "TransactionDT"}
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X = train_df[feature_cols]
y = train_df[target_col]

# Convert object/string columns to categorical dtype for LightGBM's native handling
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
for col in categorical_cols:
    X[col] = X[col].astype("category")

print(f"Training LightGBM on all {X.shape[0]:,} rows and {X.shape[1]} features...")

# Configure LightGBM classifier for full dataset training
lgb_model = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.03,
    max_depth=7,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(X, y)

# Extract and rank feature importances across all rows
feature_importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": lgb_model.feature_importances_
}).sort_values(by="importance", ascending=False)

# Select the Top N most important features (e.g., Top 25)
TOP_N = 25
top_features = feature_importance_df.head(TOP_N)["feature"].tolist()

print("\n--- Top 10 Most Important Features (Train Dataset) ---")
print(feature_importance_df.head(10))

# Save the top features whitelist locally for your chunked embedding pipeline
whitelist_path = silver_path / "top_features_whitelist.json"
with open(whitelist_path, "w") as f:
    json.dump(top_features, f)

print(f"\nSuccessfully saved top {TOP_N} feature whitelist to: {whitelist_path}")



Loading the FULL unified training dataset (590,540 rows)...


/tmp/ipykernel_1405377/3513781.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype("category")
/tmp/ipykernel_1405377/3513781.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype("category")
/tmp/ipykernel_1405377/3513781.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

/tmp/ipykernel_1405377/3513781.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype("category")


Training LightGBM on all 413,378 rows and 431 features...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.579508 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34618
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 429
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794

--- Top 10 Most Important Features (Train Dataset) ---
            feature  importance
430      DeviceInfo         339
0    TransactionAmt         1